# Environment Setup

In [10]:
!nvidia-smi

Tue Sep 29 18:06:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [11]:
!pip install unsloth bitsandbytes peft transformers datasets accelerate

In [14]:
!pip install torch
import torch
print(torch.cuda.is_available())

True


In [72]:
# From Phase-0/1
BASE_MODEL_NAME = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit"

# From Phase-2
PHASE2_ADAPTER_REPO = "JaveedHabeeb/text-to-sql-qwen2.5-coder-3b-phase2"
PHASE2_EXEC_ACC = 0.725
PHASE2_EXACT_MATCH = 0.435

# Merge the LoRA Adapter into the Base Model

In [16]:
import time
import torch
from unsloth import FastLanguageModel
from peft import PeftModel
import shutil

start = time.time()

print("Step 1: Loading base model...", flush=True)

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME.replace("-bnb-4bit", ""),
    max_seq_length=1024,
    load_in_4bit=False,
    dtype=torch.float16,
)

print(f"Base model loaded in {(time.time()-start)/60:.2f} min")

print("Step 2: Loading LoRA adapter...", flush=True)

peft_model = PeftModel.from_pretrained(
    base_model,
    PHASE2_ADAPTER_REPO
)

print(f"Adapter loaded in {(time.time()-start)/60:.2f} min")

print("Step 3: Merging LoRA weights...", flush=True)

merged_model = peft_model.merge_and_unload()

print(f"Merging completed in {(time.time()-start)/60:.2f} min")

print("Step 4: Saving merged model...", flush=True)

MERGED_DIR = "merged_model"
shutil.rmtree(MERGED_DIR, ignore_errors=True)

merged_model.save_pretrained(
    MERGED_DIR,
    safe_serialization=True,
    max_shard_size="2GB"
)

tokenizer.save_pretrained(MERGED_DIR)

print(f"Saving completed in {(time.time()-start)/60:.2f} min")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0929 18:06:53.579000 17604 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0929 18:06:53.641000 17604 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
/usr/local/lib/python3.13/dist-packages/unsloth/_gpu_init.py:376: UserWarning: torchcodec 0.11.0+cu128 is incompatible with torch 2.12.1+cu130; install a matching build with `pip install --index-url https://download.pytorch.org/whl/cu130 'torchcodec>=0.12.0'`.
  disable_torchcodec_if_broken()
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used f

🦥 Unsloth Zoo will now patch everything to make training faster!


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using

Step 1: Loading base model...
==((====))==  Unsloth 2026.9.12: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Base model loaded in 0.55 min
Step 2: Loading LoRA adapter...
Adapter loaded in 0.58 min
Step 3: Merging LoRA weights...
Merging completed in 0.58 min
Step 4: Saving merged model...
Saving completed in 1.62 min


In [17]:
import os
print("Merged model files:", os.listdir(MERGED_DIR))
assert any(f.endswith((".safetensors", ".bin")) for f in os.listdir(MERGED_DIR)), \
    "No full weight files found — merge likely failed"
assert "adapter_config.json" not in os.listdir(MERGED_DIR), \
    "adapter_config.json present — this looks like an unmerged adapter dir, not merged weights"

Merged model files: ['model-00001-of-00004.safetensors', 'merges.txt', 'special_tokens_map.json', 'model.safetensors.index.json', 'model-00003-of-00004.safetensors', 'model-00002-of-00004.safetensors', 'config.json', 'vocab.json', 'tokenizer_config.json', 'tokenizer.json', 'model-00004-of-00004.safetensors', 'generation_config.json', 'added_tokens.json', 'chat_template.jinja']


# Save Merged Model to Drive

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import shutil

# Source: existing merged model directory
MERGED_DIR = "/content/merged_model"

# Destination: Google Drive
DRIVE_MERGED_DIR = "/content/drive/MyDrive/text-to-sql-finetuning/merged_model"

# Remove old copy if it exists
if os.path.exists(DRIVE_MERGED_DIR):
    shutil.rmtree(DRIVE_MERGED_DIR)

# Copy merged model
shutil.copytree(MERGED_DIR, DRIVE_MERGED_DIR)

print("Merged model saved successfully!")
print("Location:", DRIVE_MERGED_DIR)

# Verify saved files
print("\nSaved files:")
for file in os.listdir(DRIVE_MERGED_DIR):
    print(file)

# Verify total size
total_size = sum(
    os.path.getsize(os.path.join(root, file))
    for root, _, files in os.walk(DRIVE_MERGED_DIR)
    for file in files
)

print(f"\nTotal size: {total_size / (1024**3):.2f} GB")

In [ ]:
import json

index_path = os.path.join(
    DRIVE_MERGED_DIR,
    "model.safetensors.index.json"
)

if os.path.exists(index_path):
    with open(index_path) as f:
        weight_map = json.load(f)["weight_map"]

    print("Total tensors:", len(weight_map))
    print("Model weights verified!")

else:
    print("Single-file model or index file not found.")
    print("Check the saved .safetensors files.")

# Sanity-check: merged model vs adapter-attached model on known prompts


### Funcs from Phase 0/1

In [18]:
!apt-get install -y git-lfs
!git lfs install
!git clone https://huggingface.co/datasets/minktn/spider-data
!ls spider-data

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git-lfs is already the newest version (3.4.1-1ubuntu0.4).
0 upgraded, 0 newly installed, 0 to remove and 52 not upgraded.
Git LFS initialized.
fatal: destination path 'spider-data' already exists and is not an empty directory.
README.md  spider_data


In [19]:
import json
import re
from datasets import load_dataset
spider = load_dataset("xlangai/spider")

with open("spider-data/spider_data/tables.json") as f:
    tables = json.load(f)
schema_lookup = {t["db_id"]: t for t in tables}
SPIDER_DB_ROOT = "spider-data/spider_data/database"

In [20]:
def schema_to_text(db_id, schema_lookup):
    schema = schema_lookup[db_id]
    table_names = schema["table_names_original"]
    column_names = schema["column_names_original"]
    column_types = schema["column_types"]
    pk_cols = set(schema.get("primary_keys", []))
    fk_pairs = schema.get("foreign_keys", [])
    tables_by_idx = {i: [] for i in range(len(table_names))}
    for col_idx, (tbl_idx, col_name) in enumerate(column_names):
        if tbl_idx == -1:
            continue
        col_type = column_types[col_idx]
        pk_marker = " PRIMARY KEY" if col_idx in pk_cols else ""
        tables_by_idx[tbl_idx].append(f"{col_name} {col_type}{pk_marker}")
    lines = []
    for tbl_idx, tbl_name in enumerate(table_names):
        cols = ", ".join(tables_by_idx[tbl_idx])
        lines.append(f"CREATE TABLE {tbl_name} ({cols});")
    if fk_pairs:
        for from_col_idx, to_col_idx in fk_pairs:
            from_tbl, from_col = column_names[from_col_idx]
            to_tbl, to_col = column_names[to_col_idx]
            lines.append(f"-- FK: {table_names[from_tbl]}.{from_col} -> {table_names[to_tbl]}.{to_col}")
    return "\n".join(lines)

def build_prompt(schema_text, question, gold_sql=None):
    system_msg = ("You are a text-to-SQL model. Given a database schema and a question, "
                  "output only the SQL query. Do not include explanations or markdown formatting.")
    user_msg = f"Schema:\n{schema_text}\n\nQuestion: {question}"
    messages = [{"role": "system", "content": system_msg}, {"role": "user", "content": user_msg}]
    if gold_sql is not None:
        messages.append({"role": "assistant", "content": gold_sql})
    return messages

import sqlite3

def get_db_connection(db_id):
    db_path = os.path.join(SPIDER_DB_ROOT, db_id, f"{db_id}.sqlite")
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"No sqlite file found for db_id={db_id} at {db_path}")
    return sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)

def run_sql(conn, sql_string):
    try:
        cursor = conn.cursor()
        cursor.execute(sql_string)
        return True, cursor.fetchall(), None
    except Exception as e:
        return False, [], str(e)

def execution_match(conn, generated_sql, gold_sql):
    gen_ok, gen_rows, _ = run_sql(conn, generated_sql)
    gold_ok, gold_rows, _ = run_sql(conn, gold_sql)
    if not gen_ok or not gold_ok:
        return False
    return set(tuple(r) for r in gen_rows) == set(tuple(r) for r in gold_rows)

def exact_match(generated_sql, gold_sql):
    norm = lambda s: s.strip().rstrip(";").strip().lower()
    return norm(generated_sql) == norm(gold_sql)

from tqdm.notebook import tqdm

def run_eval_harness(test_examples, generate_fn, results_path="eval_results.json"):
    per_example_results, n_exec, n_exact = [], 0, 0
    for i, ex in enumerate(tqdm(test_examples, desc="Evaluating")):
        db_id, question, gold_sql, schema_text = ex["db_id"], ex["question"], ex["gold_sql"], ex["schema_text"]
        try:
            generated_sql = generate_fn(question, schema_text)
        except Exception as e:
            generated_sql = ""
            print(f"[{i}] generate_fn raised: {e}")
        conn = get_db_connection(db_id)
        exec_ok = execution_match(conn, generated_sql, gold_sql)
        exact_ok = exact_match(generated_sql, gold_sql)
        n_exec += exec_ok; n_exact += exact_ok
        per_example_results.append({"index": i, "db_id": db_id, "question": question,
                                     "generated_sql": generated_sql, "gold_sql": gold_sql,
                                     "execution_match": exec_ok, "exact_match": exact_ok})
    n = len(test_examples)
    summary = {"execution_accuracy": n_exec / n if n else 0.0,
               "exact_match": n_exact / n if n else 0.0, "n": n}
    with open(results_path, "w") as f:
        json.dump({"summary": summary, "per_example": per_example_results}, f, indent=2)
    print("Summary:", summary)
    return {"summary": summary, "per_example": per_example_results}

def extract_sql(raw_output):
    text = raw_output.strip()
    fence = re.search(r"```(?:sql)?\s*(.*?)```", text, re.DOTALL)
    if fence:
        text = fence.group(1).strip()
    start = re.search(r"(SELECT|WITH|INSERT|UPDATE|DELETE)\b", text, re.IGNORECASE)
    if start:
        text = text[start.start():]
    if ";" in text:
        text = text.split(";")[0] + ";"
    return text.strip()

def generate_sql(question, schema_text, model, tokenizer, max_new_tokens=256):
    messages = build_prompt(schema_text, question)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    return extract_sql(tokenizer.decode(generated_ids, skip_special_tokens=True))


In [21]:
"""## Sanity-check: merged model vs adapter-attached model on known prompts

Reuses Phase 0/1/2's build_prompt / schema_to_text / generate_sql unchanged.
"""

# (Paste schema_to_text / build_prompt / extract_sql / generate_sql /
#  get_db_connection / run_sql / execution_match from Phase 0-2 here
#  unmodified — identical to text_to_sql_2.py's "Funcs from Phase-0"
#  section.)

# test_formatted is needed here (earlier than Task 4) to look up the
# schema_text/db_id for the failure-pattern questions below.
save_dir = "/content/drive/MyDrive/text-to-sql-finetuning/data"

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

test_formatted = load_jsonl(f"{save_dir}/test.jsonl")

# Named failure-pattern examples from phase1_summary.md, reused for every
# milestone/error-analysis check below (Task 1's sanity check, and again
# for Task 9-style rechecks against later variants if you extend this
# notebook).
FAILURE_PATTERN_EXAMPLES = {
    "Hallucinated/unnecessary joins": {
        "question": "What is the average expected life expectancy for countries in the region of Central Africa?",
        "gold_sql": 'SELECT avg(LifeExpectancy) FROM country WHERE Region  =  "Central Africa"',
    },
    "Self-join failures": {
        "question": "Show the names of all of the high schooler Kyle's friends.",
        "gold_sql": 'SELECT T3.name FROM Friend AS T1 JOIN Highschooler AS T2 ON T1.student_id  =  T2.id JOIN Highschooler AS T3 ON T1.friend_id  =  T3.id WHERE T2.name  =  "Kyle"',
    },
    "Set operations replaced with JOIN/WHERE": {
        "question": "What are the ids and names of all countries that either have more than 3 car makers or produce fiat model ?",
        "gold_sql": "select t1.countryid ,  t1.countryname from countries as t1 join car_makers as t2 on t1.countryid  =  t2.country group by t1.countryid having count(*)  >  3 union select t1.countryid ,  t1.countryname from countries as t1 join car_makers as t2 on t1.countryid  =  t2.country join model_list as t3 on t2.id  =  t3.maker where t3.model  =  'fiat';",
    },
    "!= vs NOT IN confusion": {
        "question": "Find the major and age of students who do not have a cat pet.",
        "gold_sql": "SELECT major ,  age FROM student WHERE stuid NOT IN (SELECT T1.stuid FROM student AS T1 JOIN has_pet AS T2 ON T1.stuid  =  T2.stuid JOIN pets AS T3 ON T3.petid  =  T2.petid WHERE T3.pettype  =  'cat')",
    },
}

def check_failure_patterns(model, tokenizer, label):
    """Re-generates SQL for the 4 named failure examples and prints a
    fixed/unchanged/broken-differently read against phase1_summary.md."""
    print(f"--- Failure-pattern recheck: {label} ---")
    out = {}
    for category, info in FAILURE_PATTERN_EXAMPLES.items():
        db_id = next((ex["db_id"] for ex in test_formatted if ex["question"] == info["question"]), None)
        schema_text = next((ex["schema_text"] for ex in test_formatted if ex["question"] == info["question"]), None)
        if db_id is None:
            print(f"[{category}] question not found in test_formatted — flagging, not silently skipping")
            continue
        generated = generate_sql(info["question"], schema_text, model, tokenizer)
        conn = get_db_connection(db_id)
        exec_ok = execution_match(conn, generated, info["gold_sql"])
        print(f"Category: {category}\n  Gold: {info['gold_sql']}\n  Generated: {generated}\n  execution_match={exec_ok}")
        out[category] = {"generated_sql": generated, "execution_match": exec_ok}
    return out

In [22]:
# Sanity check: does the merged model reproduce the same behavior as the
# Phase 2 adapter-attached model on these 4 known examples? This isn't a
# fresh accuracy read — it's a merge-correctness check. Compare this
# output by eye against Phase 2's recorded generations for these same
# 4 questions (from finetuned_v1_eval_results.json / your leading Phase 2
# checkpoint's per-example results).
merged_sanity_results = check_failure_patterns(merged_model, tokenizer, label="merged f16 model")

--- Failure-pattern recheck: merged f16 model ---
Category: Hallucinated/unnecessary joins
  Gold: SELECT avg(LifeExpectancy) FROM country WHERE Region  =  "Central Africa"
  Generated: SELECT avg(LifeExpectancy) FROM country WHERE Region  =  "Central Africa"
  execution_match=True
Category: Self-join failures
  Gold: SELECT T3.name FROM Friend AS T1 JOIN Highschooler AS T2 ON T1.student_id  =  T2.id JOIN Highschooler AS T3 ON T1.friend_id  =  T3.id WHERE T2.name  =  "Kyle"
  Generated: SELECT T3.name FROM friend AS T1 JOIN highschooler AS T2 ON T1.student_id  =  T2.id JOIN highschooler AS T3 ON T1.friend_id  =  T3.id WHERE T2.name  =  "Kyle"
  execution_match=True
Category: Set operations replaced with JOIN/WHERE
  Gold: select t1.countryid ,  t1.countryname from countries as t1 join car_makers as t2 on t1.countryid  =  t2.country group by t1.countryid having count(*)  >  3 union select t1.countryid ,  t1.countryname from countries as t1 join car_makers as t2 on t1.countryid  =  t2.co

# Convert the Merged Model to GGUF Format

In [16]:
!git clone https://github.com/ggerganov/llama.cpp

fatal: destination path 'llama.cpp' already exists and is not an empty directory.


In [17]:
!pip install gguf sentencepiece

In [18]:
import os, json, shutil, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

FP16_BASE = BASE_MODEL_NAME.replace("-bnb-4bit", "")  # unsloth/Qwen2.5-Coder-3B-Instruct
MERGED_DIR = "merged_model"
shutil.rmtree(MERGED_DIR, ignore_errors=True)

tokenizer = AutoTokenizer.from_pretrained(FP16_BASE)
base_model = AutoModelForCausalLM.from_pretrained(
    FP16_BASE, torch_dtype=torch.float16, device_map="cuda"
)

peft_model = PeftModel.from_pretrained(base_model, PHASE2_ADAPTER_REPO)
print(type(peft_model).__name__)   # expect PeftModelForCausalLM

# Fold LoRA into the base weights (W' = W + BA) and remove the wrappers
merged_model = peft_model.merge_and_unload()

merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

# Verify from the saved files
with open(f"{MERGED_DIR}/model.safetensors.index.json") as f:
    names = list(json.load(f)["weight_map"].keys())
bad = [n for n in names if "lora" in n or "base_layer" in n]
assert not bad, f"Still unmerged, e.g. {bad[:3]}"
assert "adapter_config.json" not in os.listdir(MERGED_DIR)
print(f"OK: {len(names)} tensors, none PEFT-wrapped. Example: {names[5]}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

PeftModelForCausalLM
OK: 434 tensors, none PEFT-wrapped. Example: model.layers.0.post_attention_layernorm.weight


In [21]:
!python llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} \--outfile text2sql-f16.gguf --outtype f16

INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00002.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00002.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {2048, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> F16, shape = {11008, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> F16, shape = {2048, 11008}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> F16, shape = {2048, 11008}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torc

In [22]:
import os
assert os.path.exists("text2sql-f16.gguf"), "GGUF conversion did not produce an output file"
print(f"f16 GGUF size: {os.path.getsize('text2sql-f16.gguf') / 1e9:.2f} GB")

f16 GGUF size: 6.18 GB


In [23]:
!cmake -B llama.cpp/build -S llama.cpp -DGGML_CUDA=ON

-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 0.5.0-dev
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.43.0")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found Ope

In [3]:
!cmake --build llama.cpp/build --config Release -j2 --target llama-quantize

[  2%] Built target llama-common-base
[  6%] Built target ggml-base
[  6%] Building CXX object vendor/cpp-httplib/CMakeFiles/cpp-httplib.dir/httplib.cpp.o
[  6%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/acc.cu.o
[  8%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/add-id.cu.o
[  8%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/allreduce.cu.o
[  8%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/arange.cu.o
[  8%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/argmax.cu.o
[  8%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/argsort.cu.o
[  8%] Linking CXX static library libcpp-httplib.a
[  8%] Built target cpp-httplib
[ 10%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/binbcast.cu.o
[ 10%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/clamp.cu.o
[ 10%] Building C object ggml/src/CMakeFiles/ggml-cpu.dir/ggml-cpu/ggml-cpu.c.o
[ 10%] B

In [26]:
!cmake --build llama.cpp/build --config Release -j2 --target llama-cli

[  3%] Built target ggml-base
[  4%] Built target vendor-hash
[  6%] Built target llama-common-base
[  6%] Built target cpp-httplib
[  6%] Provisioning UI assets
-- UI: HF stamp matches 'ggml-org/llama-ui|b11258', skipping HF fetch
-- UI: assets unchanged, skipping embedding
[  6%] Built target llama-ui-assets
[ 11%] Built target ggml-cpu
[ 12%] Building CXX object tools/ui/CMakeFiles/llama-ui.dir/ui.cpp.o
[ 53%] Built target ggml-cuda
[ 53%] Built target ggml
[ 66%] Built target llama
[ 66%] Building CXX object tools/mtmd/CMakeFiles/mtmd.dir/mtmd.cpp.o
[ 68%] Building CXX object tools/mtmd/CMakeFiles/mtmd.dir/mtmd-audio.cpp.o
[ 68%] Linking CXX static library libllama-ui.a
[ 68%] Built target llama-ui
[ 68%] Building CXX object tools/mtmd/CMakeFiles/mtmd.dir/mtmd-image.cpp.o
[ 68%] Building CXX object tools/mtmd/CMakeFiles/mtmd.dir/mtmd-helper.cpp.o
[ 82%] Built target llama-common
[ 82%] Building CXX object tools/mtmd/CMakeFiles/mtmd.dir/mtmd-helper-gen.cpp.o
[ 82%] Building CXX obje

In [45]:
!cmake --build llama.cpp/build --config Release -j2 --target llama-server

[  1%] Built target vendor-hash
[  4%] Built target ggml-base
[  6%] Built target llama-common-base
[  6%] Built target cpp-httplib
[  6%] Provisioning UI assets
-- UI: HF stamp matches 'ggml-org/llama-ui|b11258', skipping HF fetch
-- UI: assets unchanged, skipping embedding
[  6%] Built target llama-ui-assets
[ 11%] Built target ggml-cpu
[ 12%] Built target llama-ui
[ 53%] Built target ggml-cuda
[ 53%] Built target ggml
[ 66%] Built target llama
[ 80%] Built target llama-common
[ 95%] Built target mtmd
[ 98%] Built target server-context
[ 98%] Built target llama-server-impl
[ 98%] Building CXX object tools/server/CMakeFiles/llama-server.dir/main.cpp.o
[100%] Linking CXX executable ../../bin/llama-server
[100%] Built target llama-server


In [28]:
# Use the Kyle self-join example (a known-good Phase 2 output to compare against).
# Falls back to test_formatted[0] if it isn't found.
kyle_q = FAILURE_PATTERN_EXAMPLES["Self-join failures"]["question"]
test_ex = next((ex for ex in test_formatted if ex["question"] == kyle_q), test_formatted[0])

# Rebuild the inference prompt with the same functions used in Phases 0-2.
# add_generation_prompt=True ends the prompt at "<|im_start|>assistant\n".
TEST_PROMPT = tokenizer.apply_chat_template(
    build_prompt(test_ex["schema_text"], test_ex["question"]),
    tokenize=False,
    add_generation_prompt=True,
)

print(TEST_PROMPT)
print("Gold SQL:", test_ex["gold_sql"])

# Write to a file so newlines, quotes, and special tokens survive the shell
PROMPT_FILE = "test_prompt.txt"
with open(PROMPT_FILE, "w") as f:
    f.write(TEST_PROMPT)

<|im_start|>system
You are a text-to-SQL model. Given a database schema and a question, output only the SQL query. Do not include explanations or markdown formatting.<|im_end|>
<|im_start|>user
Schema:
CREATE TABLE Highschooler (ID number PRIMARY KEY, name text, grade number);
CREATE TABLE Friend (student_id number PRIMARY KEY, friend_id number);
CREATE TABLE Likes (student_id number PRIMARY KEY, liked_id number);
-- FK: Friend.friend_id -> Highschooler.ID
-- FK: Friend.student_id -> Highschooler.ID
-- FK: Likes.student_id -> Highschooler.ID
-- FK: Likes.liked_id -> Highschooler.ID

Question: Show the names of all of the high schooler Kyle's friends.<|im_end|>
<|im_start|>assistant

Gold SQL: SELECT T3.name FROM Friend AS T1 JOIN Highschooler AS T2 ON T1.student_id  =  T2.id JOIN Highschooler AS T3 ON T1.friend_id  =  T3.id WHERE T2.name  =  "Kyle"


In [77]:
!cmake --build /content/llama.cpp/build --target llama-bench -j 2

[  2%] Built target llama-common-base
[  6%] Built target ggml-base
[  6%] Built target cpp-httplib
[ 12%] Built target ggml-cpu
[ 64%] Built target ggml-cuda
[ 64%] Built target ggml
[ 80%] Built target llama
[ 98%] Built target llama-common
[ 98%] Building CXX object tools/llama-bench/CMakeFiles/llama-bench-impl.dir/llama-bench.cpp.o
[ 98%] Linking CXX shared library ../../bin/libllama-bench-impl.so
[ 98%] Built target llama-bench-impl
[ 98%] Building CXX object tools/llama-bench/CMakeFiles/llama-bench.dir/main.cpp.o
[100%] Linking CXX executable ../../bin/llama-bench
[100%] Built target llama-bench


In [31]:
!./llama.cpp/build/bin/llama-cli \
    -m text2sql-f16.gguf \
    -f test_prompt.txt \
    -n 128 \
    --temp 0 \h
    -ngl 99 \
    --no-warmup



Loading model... |-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/ 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b11258-ba0ba54d9
model      : text2sql-f16.gguf
ftype      : F16
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> <|im_start|>system
You are a text-to-SQL 

# Quantize the GGUF Model to Two or More Precision Levels

In [32]:
QUANT_LEVELS = ["Q8_0", "Q4_K_M"]

for level in QUANT_LEVELS:
    out_path = f"text2sql-{level}.gguf"
    !./llama.cpp/build/bin/llama-quantize text2sql-f16.gguf {out_path} {level}

variant_files = {"f16": "text2sql-f16.gguf"}
variant_files.update({level: f"text2sql-{level}.gguf" for level in QUANT_LEVELS})

file_sizes_gb = {name: os.path.getsize(path) / 1e9 for name, path in variant_files.items()}
print("File sizes (GB):")
for name, size in sorted(file_sizes_gb.items(), key=lambda kv: -kv[1]):
    print(f"  {name}: {size:.2f} GB")

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 14912 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB
version: 0.5.0-dev (build 11258, commit ba0ba54d9)
built with GNU 13.3.0 for Linux x86_64
llama_quantize: quantizing 'text2sql-f16.gguf' to 'text2sql-Q8_0.gguf' as Q8_0
llama_model_loader: loaded meta data with 29 key-value pairs and 434 tensors from text2sql-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                      general.sampling.temp f32    

In [39]:
# Smoke test each variant on the same prompt used in Task 2
for name, path in variant_files.items():
    print(f"--- {name} ---")
    !./llama.cpp/build/bin/llama-cli -m {path} -p "{TEST_PROMPT}" -n 128 --no-warmup

--- f16 ---


Loading model... |-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\| 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b11258-ba0ba54d9
model      : text2sql-f16.gguf
ftype      : F16
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> <|im_start|>system
You are a text-to-SQL model. Given a database schema and a question, output only the SQL query. Do not include explanations or markdown formatting.<|im_end|>
<|im_start|>user
Schema:
CREATE TABLE Highschooler (ID number PRIMARY KEY, name text, grade number);
CREATE TA

# Re-run the Eval Harness on Each Quantized Variant

In [68]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

In [69]:
import time, json

def llama_server_generate_sql(question, schema_text, port=LLAMA_PORT, max_new_tokens=256):
    messages = build_prompt(schema_text, question)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    resp = requests.post(
        f"http://127.0.0.1:{port}/completion",
        json={"prompt": prompt, "n_predict": max_new_tokens, "temperature": 0.0},
        timeout=120,
    )
    resp.raise_for_status()
    return extract_sql(resp.json().get("content", ""))

quant_eval_results = {}
quant_failure_pattern_results = {}
quant_latency = {}

for name, path in variant_files.items():
    print(f"=== Evaluating {name} ===")
    server_proc = start_llama_server(path)
    try:
        # abort early if the server isn't really answering
        smoke = llama_server_generate_sql("How many singers are there?",
                                          "CREATE TABLE singer (id number);")
        assert smoke.strip(), "empty output, aborting"

        t0 = time.time()
        result = run_eval_harness(
            test_subset, llama_server_generate_sql,
            results_path=f"eval_{name}_results.json",
        )
        elapsed = time.time() - t0
        quant_eval_results[name] = result["summary"]
        quant_latency[name] = elapsed / len(test_subset)
        print(f"{name}: {quant_eval_results[name]}  ({quant_latency[name]:.2f}s/example)")

        quant_failure_pattern_results[name] = {
            category: llama_server_generate_sql(
                info["question"],
                next(ex["schema_text"] for ex in test_formatted
                     if ex["question"] == info["question"]))
            for category, info in FAILURE_PATTERN_EXAMPLES.items()
        }
    finally:
        stop_llama_server(server_proc)

# save to Drive so a runtime reset doesn't lose it
with open("/content/drive/MyDrive/text-to-sql-finetuning/quant_eval_summary.json", "w") as f:
    json.dump({"accuracy": quant_eval_results, "sec_per_example": quant_latency,
               "failure_patterns": quant_failure_pattern_results}, f, indent=2)

=== Evaluating f16 ===


Evaluating:   0%|          | 0/200 [00:00<?, ?it/s]

Summary: {'execution_accuracy': 0.745, 'exact_match': 0.455, 'n': 200}
f16: {'execution_accuracy': 0.745, 'exact_match': 0.455, 'n': 200}  (1.02s/example)
=== Evaluating Q8_0 ===


Evaluating:   0%|          | 0/200 [00:00<?, ?it/s]

Summary: {'execution_accuracy': 0.735, 'exact_match': 0.455, 'n': 200}
Q8_0: {'execution_accuracy': 0.735, 'exact_match': 0.455, 'n': 200}  (0.69s/example)
=== Evaluating Q4_K_M ===


Evaluating:   0%|          | 0/200 [00:00<?, ?it/s]

Summary: {'execution_accuracy': 0.755, 'exact_match': 0.45, 'n': 200}
Q4_K_M: {'execution_accuracy': 0.755, 'exact_match': 0.45, 'n': 200}  (0.56s/example)


In [73]:
print("Quantization accuracy comparison:")
for name, summary in quant_eval_results.items():
    print(f"  {name}: exec_acc={summary['execution_accuracy']:.1%}  "
          f"exact_match={summary['exact_match']:.1%}")

if PHASE2_EXEC_ACC is not None:
    gap = abs(quant_eval_results["f16"]["execution_accuracy"] - PHASE2_EXEC_ACC)
    print(f"f16 GGUF vs Phase 2 transformers checkpoint gap: {gap:.1%} "
          f"(investigate if this is large/unexplained)")

Quantization accuracy comparison:
  f16: exec_acc=74.5%  exact_match=45.5%
  Q8_0: exec_acc=73.5%  exact_match=45.5%
  Q4_K_M: exec_acc=75.5%  exact_match=45.0%
f16 GGUF vs Phase 2 transformers checkpoint gap: 2.0% (investigate if this is large/unexplained)


# Accuracy Impact of Quantization

In [78]:
"""## Milestone Task 5 — Accuracy Impact of Quantization"""

q4_only_failures = [
    r for r in json.load(open("eval_Q4_K_M_results.json"))["per_example"]
    if not r["execution_match"]
]
f16_results_by_q = {
    r["question"]: r
    for r in json.load(open("eval_f16_results.json"))["per_example"]
}
regression_examples = [
    r for r in q4_only_failures
    if f16_results_by_q.get(r["question"], {}).get("execution_match")
][:5]
for r in regression_examples:
    print("=" * 60)
    print("Question:", r["question"])
    print("f16 generated:  ", f16_results_by_q[r["question"]]["generated_sql"])
    print("Q4_K_M generated:", r["generated_sql"])
    print("Gold:", r["gold_sql"])

Question: Which model of the car has the minimum horsepower?
f16 generated:   SELECT T1.Model FROM car_names AS T1 JOIN cars_data AS T2 ON T1.MakeId  =  T2.Id ORDER BY T2.Horsepower LIMIT 1
Q4_K_M generated: SELECT T1.Model FROM model_list AS T1 JOIN cars_data AS T2 ON T1.Model  =  T2.Make WHERE T2.Horsepower  =  (SELECT min(Horsepower) FROM cars_data)
Gold: SELECT T1.Model FROM CAR_NAMES AS T1 JOIN CARS_DATA AS T2 ON T1.MakeId  =  T2.Id ORDER BY T2.horsepower ASC LIMIT 1;
Question: Return the code of the template type that is most commonly used in documents.
f16 generated:   SELECT T1.template_type_code FROM Ref_template_types AS T1 JOIN templates AS T2 ON T1.template_type_code  =  T2.template_type_code JOIN documents AS T3 ON T2.template_id  =  T3.template_id GROUP BY T1.template_type_code ORDER BY count(*) DESC LIMIT 1
Q4_K_M generated: SELECT T1.template_type_code FROM Ref_template_types AS T1 JOIN templates AS T2 ON T1.template_type_code  =  T2.template_type_code GROUP BY T1.templ

# Benchmark Inference Latency and Memory Footprint

In [79]:
"""
Task 6 — Benchmark Inference Latency and Memory Footprint

"""

import os
import subprocess
import time
import re
import pandas as pd

N_REPEATS = 3
PROMPT_TOKENS = 512
GENERATION_TOKENS = 256

# Use the absolute path to llama-bench
LLAMA_BENCH = "/content/llama.cpp/build/bin/llama-bench"

bench_results = {}
bench_summary = []

# Verify llama-bench exists
if not os.path.isfile(LLAMA_BENCH):
    raise FileNotFoundError(
        f"llama-bench not found at: {LLAMA_BENCH}\n"
        "Build it first using:\n"
        "cmake --build /content/llama.cpp/build --target llama-bench -j 2"
    )

print(f"Using benchmark executable: {LLAMA_BENCH}")
print(f"Repeats per variant: {N_REPEATS}\n")

for name, path in variant_files.items():

    print(f"\n{'=' * 60}")
    print(f"=== Benchmarking {name} ===")
    print(f"Model: {path}")
    print(f"{'=' * 60}")

    if not os.path.isfile(path):
        print(f"WARNING: Model file not found: {path}")
        continue

    runs = []

    for i in range(N_REPEATS):

        print(f"\nRun {i + 1}/{N_REPEATS}")

        start_time = time.time()

        try:
            out = subprocess.run(
                [
                    LLAMA_BENCH,
                    "-m", path,
                    "-p", str(PROMPT_TOKENS),
                    "-n", str(GENERATION_TOKENS),
                ],
                capture_output=True,
                text=True,
                timeout=1800,  # 30-minute safety timeout
                check=False,
            )

            elapsed = time.time() - start_time

            if out.returncode != 0:
                print(f"Benchmark failed with return code {out.returncode}")
                print("STDERR:")
                print(out.stderr[-3000:])
                runs.append({
                    "run": i + 1,
                    "status": "failed",
                    "elapsed_seconds": elapsed,
                    "stdout": out.stdout,
                    "stderr": out.stderr,
                })
                continue

            print(out.stdout)

            if out.stderr.strip():
                print("Additional logs:")
                print(out.stderr[-1000:])

            runs.append({
                "run": i + 1,
                "status": "success",
                "elapsed_seconds": elapsed,
                "stdout": out.stdout,
                "stderr": out.stderr,
            })

        except subprocess.TimeoutExpired:
            print("Benchmark timed out.")
            runs.append({
                "run": i + 1,
                "status": "timeout",
                "elapsed_seconds": None,
                "stdout": "",
                "stderr": "Benchmark exceeded 30 minutes.",
            })

        except Exception as e:
            print(f"Unexpected error: {repr(e)}")
            runs.append({
                "run": i + 1,
                "status": "error",
                "elapsed_seconds": None,
                "stdout": "",
                "stderr": repr(e),
            })

    bench_results[name] = runs

    successful_runs = sum(
        1 for run in runs if run["status"] == "success"
    )

    bench_summary.append({
        "variant": name,
        "successful_runs": successful_runs,
        "total_runs": N_REPEATS,
    })

# Display run summary
summary_df = pd.DataFrame(bench_summary)

print("\n\n=== Benchmark Run Summary ===")
display(summary_df)

# Save raw benchmark output
import json

with open("/content/bench_results.json", "w") as f:
    json.dump(bench_results, f, indent=2)

print("\nRaw benchmark results saved to /content/bench_results.json")

# Peak memory:
# Capture nvidia-smi during a llama-bench run, e.g. poll
# `nvidia-smi --query-gpu=memory.used --format=csv`
# on a background thread while the benchmark subprocess above is running.

Using benchmark executable: /content/llama.cpp/build/bin/llama-bench
Repeats per variant: 3


=== Benchmarking f16 ===
Model: text2sql-f16.gguf

Run 1/3
| model                          |       size |     params | backend    | ngl |            test |                  t/s |
| ------------------------------ | ---------: | ---------: | ---------- | --: | --------------: | -------------------: |
| qwen2 3B F16                   |   5.75 GiB |     3.09 B | CUDA       |  -1 |           pp512 |      2474.13 ± 38.61 |
| qwen2 3B F16                   |   5.75 GiB |     3.09 B | CUDA       |  -1 |           tg256 |         31.04 ± 2.26 |

build: ba0ba54d9 (11258)

Additional logs:
ggml_cuda_init: found 1 CUDA devices (Total VRAM: 14912 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB


Run 2/3
| model                          |       size |     params | backend    | ngl |            test |                  t/s |
| ------------------------------ | ---------: | ------

,variant,successful_runs,total_runs
0,f16,3,3
1,Q8_0,3,3
2,Q4_K_M,3,3



Raw benchmark results saved to /content/bench_results.json


# Documented Accuracy-vs-Latency/Memory Tradeoff Table

In [81]:

"""
Milestone Task 7 — Documented Accuracy-vs-Latency/Memory Tradeoff Table

"""

import os
import pandas as pd

# Average generation throughput (tg256) from Task 6
# Values are averaged across 3 successful runs.
generation_speeds = {
    "f16": 32.50,
    "Q8_0": 53.39,
    "Q4_K_M": 64.28,
}

# Peak GPU memory was not captured during Task 6.
peak_memory = {
    "f16": None,
    "Q8_0": None,
    "Q4_K_M": None,
}

tradeoff_rows = []

for name in variant_files:

    tradeoff_rows.append({
        "variant": name,
        "file_size_gb": round(file_sizes_gb[name], 2),
        "execution_accuracy": quant_eval_results[name]["execution_accuracy"],
        "exact_match": quant_eval_results[name]["exact_match"],
        "gen_tokens_per_sec": generation_speeds.get(name),
        "peak_memory_gb": peak_memory.get(name),
    })

tradeoff_df = pd.DataFrame(tradeoff_rows)

# Calculate speed and storage improvements relative to F16
f16_speed = generation_speeds["f16"]
f16_size = file_sizes_gb["f16"]

tradeoff_df["speed_gain_vs_f16_pct"] = (
    (tradeoff_df["gen_tokens_per_sec"] / f16_speed - 1) * 100
).round(2)

tradeoff_df["storage_savings_vs_f16_pct"] = (
    (1 - tradeoff_df["file_size_gb"] / f16_size) * 100
).round(2)

print("=== Accuracy vs. Latency/Memory Tradeoff ===")
print(tradeoff_df.to_string(index=False))

# Save CSV to Google Drive
tradeoff_csv_path = (
    "/content/drive/MyDrive/text-to-sql-finetuning/comparison/"
    "phase3_quantization_tradeoff.csv"
)

os.makedirs(os.path.dirname(tradeoff_csv_path), exist_ok=True)
tradeoff_df.to_csv(tradeoff_csv_path, index=False)

print(f"\nTradeoff table saved to Drive: {tradeoff_csv_path}")

tradeoff_md_path = tradeoff_csv_path.replace(".csv", ".md")

with open(tradeoff_md_path, "w") as f:
    f.write("# Phase 3 — Quantization Tradeoff\n\n")
    f.write(tradeoff_df.to_markdown(index=False))
    f.write(
        "\n\n**Note:** Peak GPU memory was not captured during "
        "the benchmark. GGUF file size is reported separately "
        "and should not be interpreted as peak runtime memory.\n"
    )

print(f"Markdown table saved to Drive: {tradeoff_md_path}")

=== Accuracy vs. Latency/Memory Tradeoff ===
variant  file_size_gb  execution_accuracy  exact_match  gen_tokens_per_sec peak_memory_gb  speed_gain_vs_f16_pct  storage_savings_vs_f16_pct
    f16          6.18               0.745        0.455               32.50           None                   0.00                       -0.03
   Q8_0          3.29               0.735        0.455               53.39           None                  64.28                       46.75
 Q4_K_M          1.93               0.755        0.450               64.28           None                  97.78                       68.76

Tradeoff table saved to Drive: /content/drive/MyDrive/text-to-sql-finetuning/comparison/phase3_quantization_tradeoff.csv
Markdown table saved to Drive: /content/drive/MyDrive/text-to-sql-finetuning/comparison/phase3_quantization_tradeoff.md
